# PA5: Scalable, Enterprise-Grade Agentic AI Game Development
## Extension of Dino Runner base architecture with ReAct Agents, Guardrails, Memory Management, and MLflow Tracking.
**Roll Number:** 25280019

## Task 0: Workspace Initialization & Setup

In [ ]:
%pip install --upgrade typing_extensions>=4.12.0
%pip install langchain langchain_community langgraph langchain-experimental pygame
%pip install mlflow presidio-analyzer presidio-anonymizer
dbutils.library.restartPython()

## Imports, API Setup & LLM Initialization

In [ ]:
import os
import sys
import json
import time
import mlflow
import subprocess
from typing import Callable, TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.chat_models.databricks import ChatDatabricks
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

# Fix authentication for serverless compute
try:
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    os.environ['DATABRICKS_TOKEN'] = token
    os.environ['DATABRICKS_HOST'] = host
except Exception as e:
    print(f"Warning: Could not extract token: {e}")

# Initialize LLM (405B Llama)
llm = ChatDatabricks(endpoint="databricks-meta-llama-3.1-405b-instruct")

# Initialize Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# Custom Pattern Recognizer for Passwords
password_pattern = Pattern(
    name="password_pattern",
    regex=r"(?i)\b(password|passphrase|pwd|passwd|secret)\s*[:=]\s*[^\s]{6,}\b",
    score=0.85
)
password_recognizer = PatternRecognizer(
    supported_entity="PASSWORD",
    patterns=[password_pattern]
)
analyzer.registry.add_recognizer(password_recognizer)

# Custom Pattern Recognizer for API Keys (generic Developer API key pattern)
api_key_pattern = Pattern(
    name="api_key_pattern",
    regex=r"\b((?:api|db|secret|auth|client|access|token|key)_[a-zA-Z0-9_\-]{16,})\b",
    score=0.85
)
api_key_recognizer = PatternRecognizer(
    supported_entity="API_KEY",
    patterns=[api_key_pattern]
)
analyzer.registry.add_recognizer(api_key_recognizer)

## Game State Definition

In [ ]:
class GameState(TypedDict):
    director_messages: List[BaseMessage]
    architect_messages: List[BaseMessage]
    engineer_code: str
    qa_feedback: List[BaseMessage]
    current_actor: str
    iteration: int
    iteration_score: List[int]
    latencies: dict  # To track latency for MLflow
    satisfied: bool

## Task 1: ReAct Agent Architecture & Tool Integration
We develop a custom `CodeInterpreterTool` that handles headless Pygame execution using the dummy video driver, captures compilation and runtime outputs, and handles timeout errors safely.

In [ ]:
@tool
def code_interpreter_tool(code: str) -> str:
    """
    Executes the generated Python code block headlessly to test for syntax, import, or runtime errors.
    Returns the execution trace (stdout/stderr) and exit status.
    """
    import os
    import sys
    import subprocess
    
    test_filename = "temp_test_code.py"
    with open(test_filename, "w", encoding="utf-8") as f:
        f.write(code)
        
    # Configure headless execution for pygame (so it doesn't fail on displayless Databricks nodes)
    env = os.environ.copy()
    env["SDL_VIDEODRIVER"] = "dummy"
    env["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
    
    try:
        # Run code for at most 5 seconds to test initialization and game loop startup
        result = subprocess.run(
            [sys.executable, test_filename],
            capture_output=True,
            text=True,
            env=env,
            timeout=5
        )
        output = f"Exit Code: {result.returncode}\nStdout: {result.stdout}\nStderr: {result.stderr}"
        if result.returncode == 0:
            return f"SUCCESS: Code executed cleanly with exit code 0.\n{output}"
        else:
            return f"FAILURE: Code crashed during execution.\n{output}"
            
    except subprocess.TimeoutExpired as e:
        # Timeout is expected behavior because Pygame has an infinite loop while running
        stdout = e.stdout.decode() if e.stdout else ""
        stderr = e.stderr.decode() if e.stderr else ""
        return (
            "SUCCESS: Code successfully initialized and entered Pygame main loop (terminated via timeout as expected).\n"
            f"Stdout: {stdout}\nStderr: {stderr}"
        )
    except Exception as e:
        return f"FAILURE: Execution process error: {str(e)}"
    finally:
        if os.path.exists(test_filename):
            try:
                os.remove(test_filename)
            except Exception:
                pass

## Task 2: Implement Nodes & Guardrails

In [ ]:
def director_node(state: GameState):
    start_time = time.time()
    director_msg = input("Awaiting Director Prompt: ")
    
    # Analyze and redact PII entities (standard + custom password & api_key)
    results = analyzer.analyze(
        text=director_msg, 
        entities=["EMAIL_ADDRESS", "IP_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "PASSWORD", "API_KEY"], 
        language='en'
    )
    anonymized = anonymizer.anonymize(text=director_msg, analyzer_results=results)
    safe_text = anonymized.text
    
    if safe_text != director_msg:
        report = {
            "original": director_msg, 
            "redacted": safe_text, 
            "entities": [res.entity_type for res in results]
        }
        with open("pii_redaction_report.json", "w") as f:
            json.dump(report, f, indent=4)
        print("\n[Guardrail] PII detected and redacted. Report saved to 'pii_redaction_report.json'.")
        
    latencies = state.get("latencies", {})
    latencies["director"] = time.time() - start_time
    
    # Initialize history list or append to existing list
    history = state.get("director_messages", [])
    return {
        "director_messages": history + [HumanMessage(content=safe_text)], 
        "current_actor": "director", 
        "latencies": latencies
    }

In [ ]:
def architect_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0) + 1
    print(f"\n========== ITERATION {current_iter} ==========")
    
    # Retrieve director input (and all historical feedback)
    director_content = "\n".join([m.content for m in state.get("director_messages", [])])
    
    system_msg = SystemMessage(content=(
        "You are an expert Software Architect. Design the Dino Runner game system architecture. "
        "Create a detailed plan detailing object-oriented layout and state definitions. "
        "CRITICAL REQUIREMENTS:\n"
        "1. Strictly apply OOP principles (separate classes for Dino, Obstacles, and Score).\n"
        "2. Include basic functionality from PA4: jump, fall, keydown ducking, keyup restore standing height, and game active flag.\n"
        "3. Day/Night Cycle: background color dynamically updates over time in the event loop.\n"
        "4. Clustered Obstacles: Obstacles (e.g. multiple cacti) can spawn in clusters or groups (e.g. spawn 1-3 obstacles together).\n"
        "5. Speed Increase: Game/Dino speed increases gradually as score increments.\n"
        "6. High-Score Persistence: The highest score is saved/retrieved from a file (e.g., highscore.txt) to persist across launches.\n"
        "Output the architectural requirements clearly."
    ))
    
    response = llm.invoke([system_msg, HumanMessage(content=f"Design Requirements & Feedback:\n{director_content}")])
    
    latencies = state.get("latencies", {})
    latencies[f"architect_iter_{current_iter}"] = time.time() - start_time
    
    history = state.get("architect_messages", [])
    return {
        "architect_messages": history + [response], 
        "current_actor": "architect", 
        "latencies": latencies
    }

In [ ]:
# Create ReAct agent for the Engineer with the code interpreter tool
engineer_agent = create_react_agent(
    llm, 
    tools=[code_interpreter_tool],
    state_modifier=(
        "You are a Senior Game Developer. Implement a Pygame Dino Runner game based on the Architect's design.\n"
        "You MUST write and test your code using the code_interpreter_tool.\n"
        "If execution fails, analyze the stdout/stderr returned by the tool, identify the bug, and write corrected code (Tool Fallback).\n"
        "Ensure all requirements are fully realized (OOP, Day/Night change background, Clustered obstacles, Speed increase, High-Score persistence file).\n"
        "Once tested successfully, provide the final tested code wrapped in a ```python ... ``` block."
    )
)

def engineer_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0) + 1
    
    architect_design = state.get("architect_messages", [])[-1].content
    qa_feedback = state.get("qa_feedback", [])
    
    qa_context = ""
    if qa_feedback:
        qa_context = f"\nPrevious QA Feedback/Bugs to Fix:\n{qa_feedback[-1].content}"
        
    prompt = f"Architect Design:\n{architect_design}\n{qa_context}\n\nGenerate and test the Python code. Ensure it uses standard Python libraries + Pygame only."
    
    res = engineer_agent.invoke({"messages": [HumanMessage(content=prompt)]})
    final_output = res["messages"][-1].content
    
    # Extract code block
    code = final_output
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0].strip()
    elif "```" in code:
        code = code.split("```")[1].split("```")[0].strip()
        
    latencies = state.get("latencies", {})
    latencies[f"engineer_iter_{current_iter}"] = time.time() - start_time
    
    return {
        "engineer_code": code, 
        "current_actor": "engineer", 
        "iteration": current_iter, 
        "latencies": latencies
    }

In [ ]:
qa_agent = create_react_agent(
    llm, 
    tools=[code_interpreter_tool],
    state_modifier=(
        "You are an expert QA Engineer. Evaluate the generated Pygame code against requirements.\n"
        "Use the code_interpreter_tool to compile and run the code headlessly.\n"
        "Verify: 1. OOP design. 2. Jump/duck physics. 3. Clustered Obstacles. 4. Day/Night background. 5. Speed increase. 6. High-score persistence file.\n"
        "Provide a comprehensive, objective report pointing out any failures or validating successes."
    )
)

def qa_node(state: GameState):
    start_time = time.time()
    current_iter = state.get("iteration", 0)
    
    architect_design = state.get("architect_messages", [])[-1].content
    engineer_code = state.get("engineer_code", "")
    
    prompt = f"Architect Design:\n{architect_design}\n\nGenerated Code to Evaluate:\n```python\n{engineer_code}\n```"
    res = qa_agent.invoke({"messages": [HumanMessage(content=prompt)]})
    qa_report = res["messages"][-1].content
    
    latencies = state.get("latencies", {})
    latencies[f"qa_iter_{current_iter}"] = time.time() - start_time
    
    history = state.get("qa_feedback", [])
    return {
        "qa_feedback": history + [HumanMessage(content=qa_report)], 
        "current_actor": "qa", 
        "latencies": latencies
    }

class ScoreOutput(BaseModel):
    score: int = Field(description="Integrity/completeness score from 1 to 10")

def score_node(state: GameState):
    qa_report = state.get("qa_feedback", [])[-1].content
    structured_llm = llm.with_structured_output(ScoreOutput)
    
    try:
        res = structured_llm.invoke([
            SystemMessage(content="Rate the code completeness and compliance based on the QA report. Return a structured score 1 to 10."),
            HumanMessage(content=qa_report)
        ])
        score = res.score
    except Exception:
        score = 5  # Safe default if parsing fails
        
    print(f"\n[Scorer Node] Assigned Score: {score}/10")
    
    scores = state.get("iteration_score", [])
    return {
        "iteration_score": scores + [score], 
        "current_actor": "scorer"
    }

## Task 3: Advanced Memory & Context Management
We implement a context summarizer middleware to compress message history once a token threshold is breached or when we hit the mandatory minimum 3 iterations.

In [ ]:
def summarizer_middleware(state: GameState):
    current_iter = state.get("iteration", 0)
    architect_msgs = state.get("architect_messages", [])
    qa_msgs = state.get("qa_feedback", [])
    
    # Estimate total token count (approx. 4 chars per token)
    total_chars = sum(len(m.content) for m in architect_msgs) + sum(len(m.content) for m in qa_msgs)
    token_est = total_chars // 4
    
    # Predefined threshold (e.g., 2000 tokens)
    threshold = 2000
    
    # We must also force summarization on/after iteration 2 to satisfy Task 3.1 requirement (3 iterations min, demonstrate summarization)
    if token_est > threshold or current_iter >= 2:
        print(f"\n[Summarizer Middleware] Context tokens ({token_est}) exceeded threshold ({threshold}) or iteration constraint reached. Condensing history...")
        
        # Summarize Architect Designs
        if architect_msgs:
            history_str = "\n\n".join([f"Design {i+1}:\n{m.content}" for i, m in enumerate(architect_msgs)])
            arch_summary_prompt = f"Summarize this software design history. Consolidate into a single concise final list of game system designs:\n\n{history_str}"
            arch_res = llm.invoke([SystemMessage(content="You are a context compression assistant."), HumanMessage(content=arch_summary_prompt)])
            new_architect = [AIMessage(content=f"[CONSOLIDATED ARCHITECT DESIGN SUMMARY]:\n{arch_res.content}")]
        else:
            new_architect = architect_msgs
            
        # Summarize QA Feedback
        if qa_msgs:
            feedback_str = "\n\n".join([f"Feedback {i+1}:\n{m.content}" for i, m in enumerate(qa_msgs)])
            qa_summary_prompt = f"Summarize this QA analysis history. Retain only outstanding bugs, validation failures, and design deviations:\n\n{feedback_str}"
            qa_res = llm.invoke([SystemMessage(content="You are a context compression assistant."), HumanMessage(content=qa_summary_prompt)])
            new_qa = [HumanMessage(content=f"[CONSOLIDATED QA FEEDBACK SUMMARY]:\n{qa_res.content}")]
        else:
            new_qa = qa_msgs
            
        print("[Summarizer Middleware] Context successfully summarized and replaced.")
        return {
            "architect_messages": new_architect,
            "qa_feedback": new_qa,
            "current_actor": "summarizer"
        }
        
    return {"current_actor": "summarizer"}

In [ ]:
def director_review_node(state: GameState):
    print("\n================== DIRECTOR HITL REVIEW ==================")
    latest_score = state.get("iteration_score", [])[-1]
    current_iter = state.get("iteration", 0)
    print(f"Iteration: {current_iter} | Current Score: {latest_score}/10")
    
    # Task 3.1 requires executing at least 3 iterations
    if current_iter < 3:
        print(f"[HITL] Iteration {current_iter}/3. Forcing another iteration to demonstrate context summarization.")
        satisfied_choice = 'n'
        feedback = "Forcing iteration to meet the 3-iteration demonstration requirement."
    else:
        satisfied_choice = input("Are you satisfied with the generated game code? (y/n): ").strip().lower()
        if satisfied_choice == 'y':
            feedback = ""
        else:
            feedback = input("Please enter your custom feedback/revisions for the Architect: ").strip()
            
    if satisfied_choice == 'y':
        return {"satisfied": True, "current_actor": "director_feedback"}
    else:
        history = state.get("director_messages", [])
        # Append the new director feedback to director_messages history
        new_feedback_msg = HumanMessage(content=f"[Director Feedback - Iteration {current_iter}]: {feedback}")
        return {
            "satisfied": False, 
            "director_messages": history + [new_feedback_msg],
            "current_actor": "director_feedback"
        }

## Task 3: StateGraph & Checkpoint Compiler
We register the nodes and setup conditional routing. The graph is compiled with persistent checkpointers and interrupts before every individual agent (`architect`, `engineer`, `qa`, `director_review`).

In [ ]:
def routing_edge(state: GameState):
    if state.get("satisfied", False):
        return END
    return "architect"

workflow = StateGraph(GameState)

# Register Nodes
workflow.add_node("director", director_node)
workflow.add_node("architect", architect_node)
workflow.add_node("engineer", engineer_node)
workflow.add_node("qa", qa_node)
workflow.add_node("scorer", score_node)
workflow.add_node("summarizer", summarizer_middleware)
workflow.add_node("director_feedback", director_review_node)

# Add Edges
workflow.add_edge(START, "director")
workflow.add_edge("director", "architect")
workflow.add_edge("architect", "engineer")
workflow.add_edge("engineer", "qa")
workflow.add_edge("qa", "scorer")
workflow.add_edge("scorer", "summarizer")
workflow.add_edge("summarizer", "director_feedback")

# Routing after HITL node
workflow.add_conditional_edges("director_feedback", routing_edge, {END: END, "architect": "architect"})

# Configure memory checkpointing and agent interrupt boundaries
memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["architect", "engineer", "qa", "director_feedback"]
)

## Task 4: System Execution & MLflow Tracking
We execute the compiled graph. When interrupted, the user is notified of the boundary and can view progress. After the HITL node executes, a child run is logged to MLflow capturing the iteration details, latencies, groundedness metrics, and game script.

In [ ]:
def calculate_groundedness(plan: str, code: str) -> float:
    """
    LLM-as-a-judge metric to evaluate how accurately the generated code implements the design requirements.
    Returns a score between 0.0 and 1.0.
    """
    prompt = (
        "Evaluate the groundedness (accuracy of implementation) of the Python Pygame code against the Architect's Plan.\n"
        "Check if OOP design, Day/Night changes, Clustered obstacles, Speed increases, and High score saving are present.\n"
        "Plan:\n"
        f"{plan}\n\n"
        "Code:\n"
        f"{code}\n\n"
        "Rate groundedness on a scale of 0.0 (completely ungrounded) to 1.0 (perfectly grounded/compliant).\n"
        "Your response must be ONLY a single floating-point number, e.g., 0.95"
    )
    try:
        response = llm.invoke([SystemMessage(content="You are a strict code evaluator."), HumanMessage(content=prompt)])
        return float(response.content.strip())
    except Exception:
        return 0.5

# Set thread session config for checkpointer
config = {"configurable": {"thread_id": "dino_runner_pa5_session"}}

# Define MLflow Experiment
mlflow.set_experiment("/Users/25280019/PA5_Experiment")

# Run execution
print("--- Starting PA5 LangGraph Workflow ---")
initial_state = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": "",
    "qa_feedback": [],
    "current_actor": "",
    "iteration": 0,
    "iteration_score": [],
    "latencies": {},
    "satisfied": False
}

with mlflow.start_run(run_name="PA5_Pipeline_Run") as parent_run:
    # Initialize or resume graph stream
    state_snap = app.get_state(config)
    if not state_snap.values:
        # Start graph stream
        for event in app.stream(initial_state, config=config):
            pass
    
    while True:
        state_snap = app.get_state(config)
        
        # Break if the graph execution is completed
        if not state_snap.next:
            break
            
        next_agent = state_snap.next[0]
        print(f"\n[HITL Pause] Halting execution before node: '{next_agent}'")
        
        # Display context updates before resuming
        if next_agent == "architect" and state_snap.values.get("director_messages"):
            print(f"Latest Director Prompt: '{state_snap.values['director_messages'][-1].content}'")
        elif next_agent == "engineer" and state_snap.values.get("architect_messages"):
            print(f"Latest Architect Design (truncated): '{state_snap.values['architect_messages'][-1].content[:200]}...'")
        elif next_agent == "qa" and state_snap.values.get("engineer_code"):
            print("Code generated successfully. Ready to run QA.")
        elif next_agent == "director_feedback" and state_snap.values.get("iteration_score"):
            print(f"Scorer assigned score: {state_snap.values['iteration_score'][-1]}/10")
            
        input("Press Enter to resume execution and proceed...")
        
        # Resume the graph
        for event in app.stream(None, config=config):
            pass
            
        # Logging iteration results after director_feedback runs
        updated_state = app.get_state(config).values
        if updated_state.get("current_actor") == "director_feedback":
            current_iter = updated_state.get("iteration", 0)
            score = updated_state.get("iteration_score", [])[-1]
            code = updated_state.get("engineer_code", "")
            plan = updated_state.get("architect_messages", [])[-1].content if updated_state.get("architect_messages") else ""
            
            # Compute groundedness score
            groundedness = calculate_groundedness(plan, code)
            
            # Start Nested MLflow Run for this iteration
            with mlflow.start_run(run_name=f"Iteration_{current_iter}", nested=True):
                mlflow.log_metric("groundedness", groundedness)
                mlflow.log_metric("score", score)
                
                # Fetch latencies
                latencies = updated_state.get("latencies", {})
                arch_latency = latencies.get(f"architect_iter_{current_iter}", 0)
                eng_latency = latencies.get(f"engineer_iter_{current_iter}", 0)
                qa_latency = latencies.get(f"qa_iter_{current_iter}", 0)
                
                mlflow.log_metric("latency_architect", arch_latency)
                mlflow.log_metric("latency_engineer", eng_latency)
                mlflow.log_metric("latency_qa", qa_latency)
                mlflow.log_metric("iteration_latency", arch_latency + eng_latency + qa_latency)
                
                # Log code as artifact
                mlflow.log_text(code, f"dino_runner_iter_{current_iter}.py")
                
            print(f"\n[MLflow] Logged Iteration {current_iter} run. Groundedness={groundedness}, Latency={arch_latency+eng_latency+qa_latency:.2f}s")

    # Extract final code and write out
    final_state = app.get_state(config).values
    final_code = final_state.get("engineer_code", "")
    with open("25280019_dino_runner.py", "w", encoding="utf-8") as f:
        f.write(final_code)
    print("\n--- Pipeline Completed Successfully! ---")
    print("Final game script written to '25280019_dino_runner.py'")